In [1]:
from dataclasses import dataclass

@dataclass
class Ride:
    lpep_pickup_datetime: int  # epoch milliseconds
    lpep_dropoff_datetime: int  # epoch milliseconds
    PULocationID: int
    DOLocationID: int
    passenger_count: int
    trip_distance: float
    tip_amount: float
    total_amount: float

In [2]:
import json

def ride_deserializer(data):
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return Ride(**ride_dict)

In [3]:
from kafka import KafkaConsumer

server = 'redpanda:9092'
topic_name = 'green-trips'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer=ride_deserializer
)

In [ ]:
from datetime import datetime

print(f"Listening to {topic_name}...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.lpep_pickup_datetime / 1000)
    print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, "
          f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
          f"pickup={pickup_dt}")
    count += 1
    if count >= 10:
        print(f"\n... received {count} messages so far (stopping after 10 for demo)")
        break

consumer.close()

In [4]:
trips_greater_than_5 = 0

print(f"Listening to {topic_name}...")

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='distance-counter',
    value_deserializer=ride_deserializer,
)

try:
    for message in consumer:
        ride = message.value
        
        if ride.trip_distance > 5.0:
            trips_greater_than_5 += 1
except KeyboardInterrupt:
    print("Stopped by user")
finally:
    consumer.close()

print(f"\nFinal Count: {trips_greater_than_5} trips have a distance > 5.0")


Listening to green-trips...
Stopped by user

Final Count: 8506 trips have a distance > 5.0
